# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
links = fetch_website_links("https://promptql.io/")
links

['/',
 '/product',
 '/why-promptql-works',
 '/solutions/financial-services',
 '/solutions/healthcare',
 '/solutions/retail',
 '/case-studies',
 'https://promptql.io/blog',
 '/events',
 'https://promptql.io/docs/index/',
 '/learn/videos',
 '/pricing',
 '/book-demo',
 'https://www.reddit.com/r/PromptQL/',
 'https://prompt.ql.app/login?pg=home&cta=nav',
 '/product',
 '/why-promptql-works',
 '/solutions/financial-services',
 '/solutions/healthcare',
 '/solutions/retail',
 '/case-studies',
 'https://promptql.io/blog',
 '/events',
 'https://promptql.io/docs/index/',
 '/learn/videos',
 '/pricing',
 '/book-demo',
 'https://www.reddit.com/r/PromptQL/',
 'https://prompt.ql.app/login?pg=home&cta=hero',
 '/book-demo',
 'https://cdn.promptql.io/apps/desktop/v1.9.0/PromptQL-1.9.0-arm64.dmg',
 'https://apps.apple.com/in/app/promptql/id6757606292',
 'https://play.google.com/store/apps/details?id=app.ql.prompt',
 '#wiki',
 '#wiki',
 '#wiki',
 '#wiki',
 'https://promptql.io/blog/i-am-always-in-a-flow-st

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [12]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
print(get_links_user_prompt("https://promptql.io/"))


Here is the list of links on the website https://promptql.io/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/product
/why-promptql-works
/solutions/financial-services
/solutions/healthcare
/solutions/retail
/case-studies
https://promptql.io/blog
/events
https://promptql.io/docs/index/
/learn/videos
/pricing
/book-demo
https://www.reddit.com/r/PromptQL/
https://prompt.ql.app/login?pg=home&cta=nav
/product
/why-promptql-works
/solutions/financial-services
/solutions/healthcare
/solutions/retail
/case-studies
https://promptql.io/blog
/events
https://promptql.io/docs/index/
/learn/videos
/pricing
/book-demo
https://www.reddit.com/r/PromptQL/
https://prompt.ql.app/login?pg=home&cta=hero
/book-demo
https://cdn.promptql.io/apps/desktop/v1.9.0/PromptQL-1.9.0-arm64.dmg
https://apps.apple.com/in/app/promptql/i

In [15]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}  
        ],
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [16]:
select_relevant_links("https://promptql.io/")

{'links': [{'type': 'about page', 'url': 'https://promptql.io/about'},
  {'type': 'careers page', 'url': 'https://promptql.io/careers'},
  {'type': 'case studies page', 'url': 'https://promptql.io/case-studies'},
  {'type': 'blog page', 'url': 'https://promptql.io/blog'},
  {'type': 'events page', 'url': 'https://promptql.io/events'},
  {'type': 'learn videos page', 'url': 'https://promptql.io/learn/videos'},
  {'type': 'pricing page', 'url': 'https://promptql.io/pricing'},
  {'type': 'book a demo page', 'url': 'https://promptql.io/book-demo'},
  {'type': 'product page', 'url': 'https://promptql.io/product'},
  {'type': 'why PromptQL Works page',
   'url': 'https://promptql.io/why-promptql-works'},
  {'type': 'solutions page - financial services',
   'url': 'https://promptql.io/solutions/financial-services'},
  {'type': 'solutions page - healthcare',
   'url': 'https://promptql.io/solutions/healthcare'},
  {'type': 'solutions page - retail',
   'url': 'https://promptql.io/solutions/ret

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
select_relevant_links("https://promptql.io/")

Selecting relevant links for https://promptql.io/ by calling gpt-5-nano
Found 21 relevant links


{'links': [{'type': 'about page', 'url': 'https://promptql.io/about'},
  {'type': 'careers page', 'url': 'https://promptql.io/careers'},
  {'type': 'case studies page', 'url': 'https://promptql.io/case-studies'},
  {'type': 'blog', 'url': 'https://promptql.io/blog'},
  {'type': 'events page', 'url': 'https://promptql.io/events'},
  {'type': 'product page', 'url': 'https://promptql.io/product'},
  {'type': 'solutions page - financial services',
   'url': 'https://promptql.io/solutions/financial-services'},
  {'type': 'solutions page - healthcare',
   'url': 'https://promptql.io/solutions/healthcare'},
  {'type': 'solutions page - retail',
   'url': 'https://promptql.io/solutions/retail'},
  {'type': 'solutions page - AI analysts',
   'url': 'https://promptql.io/solutions/ai-analysts'},
  {'type': 'solutions page - data analytics',
   'url': 'https://promptql.io/solutions/data-analytics'},
  {'type': 'solutions page - go-to-market',
   'url': 'https://promptql.io/solutions/gtm'},
  {'typ

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://promptql.io/"))

Selecting relevant links for https://promptql.io/ by calling gpt-5-nano
Found 6 relevant links
## Landing Page:

PromptQL | Multiplayer AI with a Wiki.

Product
Platform
Core features and capabilities
Architecture
How and why it works
Solutions
For financial services
Transform data workflow in FinServe
For healthcare
AI analysts for health data workflows
For retail
AI analysts for retail data workflows
Case Studies
Resources
Blog
Insights, updates & stories
Events
Talks, conferences & meetups
Docs
Guides & developer resources
Learn
Explore and Watch PromptQL Videos
Pricing
Book demo
r/PromptQL
Try PromptQL
Product
Platform
Core features and capabilities
Architecture
How and why it works
Solutions
For financial services
Transform data workflow in FinServe
For healthcare
AI analysts for health data workflows
For retail
AI analysts for retail data workflows
Case Studies
Resources
Blog
Insights, updates & stories
Events
Talks, conferences & meetups
Docs
Guides & developer resources
Learn
E

In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
get_brochure_user_prompt("PromptQL", "https://promptql.io/")

Selecting relevant links for https://promptql.io/ by calling gpt-5-nano
Found 23 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: PromptQL\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nPromptQL | Multiplayer AI with a Wiki.\n\nProduct\nPlatform\nCore features and capabilities\nArchitecture\nHow and why it works\nSolutions\nFor financial services\nTransform data workflow in FinServe\nFor healthcare\nAI analysts for health data workflows\nFor retail\nAI analysts for retail data workflows\nCase Studies\nResources\nBlog\nInsights, updates & stories\nEvents\nTalks, conferences & meetups\nDocs\nGuides & developer resources\nLearn\nExplore and Watch PromptQL Videos\nPricing\nBook demo\nr/PromptQL\nTry PromptQL\nProduct\nPlatform\nCore features and capabilities\nArchitecture\nHow and why it works\nSolutions\nFor financial services\nTransform data workflow in FinServe\nFor healthcare\nAI analysts for health data workflows\nFor retail\nAI ana

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("PromptQL", "https://promptql.io/")

Selecting relevant links for https://promptql.io/ by calling gpt-5-nano
Found 23 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# PromptQL Brochure

---

## About PromptQL

PromptQL is pioneering the future of reliable AI as the creators of "Multiplayer AI with a Wiki"—an AI-native Slack experience that seamlessly connects your data with your team. Built by the Hasura team, recognized for the Hasura GraphQL Engine and Data Delivery Network trusted by thousands, PromptQL is designed to be the de facto data access layer for AI.

Unlike most enterprise AI tools that struggle with accuracy and trust, PromptQL prioritizes precision, reliability, and contextual understanding, enabling businesses to confidently deploy AI in mission-critical workflows. According to Forbes and an MIT report, while 95% of enterprise AI fails, PromptQL is proudly in the 5% that succeed.

---

## What PromptQL Does

- **Contextual AI Collaboration:** Builds and maintains a shared context in a collaborative wiki as your team works, getting smarter with every conversation.
- **Centralized Coordination:** Coordinates across people, tools, and data sources in one conversation thread, providing clarity and reducing friction.
- **Custom AI Agents:** Leverages advanced AI models such as Codex, Claude Code, and integrations with Cloudflare among others, empowering custom automation.

---

## Solutions Tailored for Industries

- **Financial Services:** Transform data workflows with AI analysts that enhance accuracy and speed.
- **Healthcare:** Streamline health data workflows using AI to support analysts in managing complex datasets.
- **Retail:** Enable AI-powered analysis to handle dynamic retail data, improving decisions and operations.

Each solution integrates with existing workflows, ensuring AI acts as an insightful partner rather than a standalone tool.

---

## Trusted at Scale

PromptQL serves diverse enterprise clients, assisting in complex decision-making processes involving multiple stakeholders, systems, and data points. Its proven reliability makes it a trusted partner in finance, healthcare, retail, and beyond.

---

## Company Culture & Vision

PromptQL champions an **engineering-first and data-centric mindset** inherited from Hasura’s legacy. The company deeply values:

- **Innovation:** Pioneering new frontiers in AI with a commitment to precision and trust.
- **Collaboration:** Promoting teamwork and collective intelligence by connecting people and data seamlessly.
- **Reliability:** Delivering enterprise-grade solutions that meet the highest standards of accuracy and trustworthiness.
- **Transparency:** Building a shared knowledge base that grows smarter with every interaction.

---

## Careers at PromptQL

Join PromptQL to shape the future of **reliable AI**. The company seeks passionate individuals eager to:

- Work on cutting-edge AI platforms that actually succeed in enterprise environments.
- Collaborate in a culture that values trust, precision, and engineering excellence.
- Solve hard problems in data access, AI reasoning, and workflow automation.
- Build scalable, impactful solutions trusted by leading global firms.

Open roles and opportunities span product development, AI research, software engineering, and more.

---

## Get Started with PromptQL

- Experience **PromptQL for free during preview**
- Book a personalized **demo**
- Download apps on **Mac App Store**, **iOS App Store**, and **Google Play**

Explore further resources including technical documentation, case studies, and blog insights at [PromptQL Website](https://promptql.com).

---

## Summary

PromptQL is revolutionizing AI in enterprises by creating an intelligent, collaborative platform that understands and acts on your data with unprecedented accuracy and trust. Whether you are a customer, investor, or potential recruit, PromptQL offers a unique opportunity to engage with the next generation of AI-powered workflow transformation.

---

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("PromptQL", "https://promptql.io/")

Selecting relevant links for https://promptql.io/ by calling gpt-5-nano
Found 6 relevant links


# PromptQL: Multiplayer AI with a Wiki

---

## About PromptQL

PromptQL is an innovative AI-native platform developed by the team behind Hasura, creators of the Hasura GraphQL Engine and Hasura Data Delivery Network, trusted by thousands worldwide. It represents the future of reliable enterprise AI, focusing on transforming how businesses access, understand, and act on their data with precision.

**The Mission:**  
To build the de facto data access layer for AI — not simply a tool for data retrieval but a platform that deeply comprehends your data and ensures accuracy, trust, and reliability, which are often the largest barriers to enterprise AI adoption.

PromptQL leverages a multiplayer AI approach combined with a live, evolving wiki that builds shared context with every conversation. This makes the system smarter over time, expertly coordinating between people, data sources, and tools within a single thread, ensuring collaborative, informed decision-making.

---

## Core Features & Capabilities

- **Multiplayer AI:** Enables seamless collaboration across teams and tools within one conversation thread, automatically pulling relevant context from a shared company wiki.
- **Data Contextualization:** Combines AI with a living knowledge base (wiki) that evolves as teams work, enhancing the relevance and accuracy of responses.
- **Integrated Workflow:** Supports complex decisions spanning multiple stakeholders and systems by orchestrating interactions intelligently.
- **Trusted at scale:** Utilized by major platforms such as Codex, Claude Code, Cloudflare, and custom AI agents, highlighting robust, scalable enterprise applications.

---

## Industry Solutions

PromptQL tailors its AI-powered data workflows across key industries:

- **Financial Services:** Transform data workflows in financial services to streamline operations and decision-making.
- **Healthcare:** Deploy AI analysts to interpret and optimize health data workflows.
- **Retail:** Use AI analysts to enhance retail data workflows for improved insight and customer experience.

---

## Why PromptQL Succeeds

According to a report highlighted by Forbes, 95% of enterprise AI initiatives fail. PromptQL belongs to the elite 5% that succeed by focusing on:

- Accuracy and trustworthiness in AI outputs.
- Integration of LLMs (large language models) with strong engineering foundations.
- Building a platform, not just a product, that deeply understands and interacts with business data.

The engineering-first mindset drawn from Hasura’s success in data engineering is a core differentiator.

---

## Company Culture & Vision

- **Engineering-Driven Innovation:** With roots in pioneering GraphQL and data delivery networks, PromptQL embodies a problem-solving culture driven by deep technical expertise.
- **Collaborative & Transparent:** The multiplayer AI wiki mirrors their values of teamwork and shared knowledge, promoting transparency and collective intelligence.
- **Customer-Centric:** PromptQL partners with enterprises to enhance real-world workflows, supporting trusted data interactions that drive better outcomes.
- **Future-Focused:** Committed to becoming the de facto AI data access layer, continuously evolving to meet sophisticated enterprise needs.

---

## Careers at PromptQL

Join PromptQL to be part of a cutting-edge team transforming enterprise AI. The company offers opportunities to:

- Work on pioneering AI data workflows integrated within business applications.
- Collaborate with experts from Hasura’s acclaimed engineering teams.
- Build solutions trusted by global leaders in finance, healthcare, and retail.
- Grow in a fast-paced, innovative culture focused on solving complex problems with reliable AI.

If you want to build the future of reliable AI that genuinely works at scale, PromptQL welcomes talented engineers, product developers, AI specialists, and data experts to explore current openings.

---

## Get Started

- **Try for free during the preview phase.**
- **Book a personalized demo to see PromptQL in action.**
- **Available on Mac App Store, iOS App Store, and Google Play.**

---

## Contact & Additional Resources

- Explore detailed **product documentation** and **developer guides**   
- Stay updated with insights, stories, events, and blog posts on AI and data workflows  
- Join community conversations on r/PromptQL  

**Website:** [promptql.com](https://promptql.com)  
**Social:** @PromptQL  

---

Empower your team with PromptQL, where AI and collaborative knowledge converge to transform data-driven decisions.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")